In [3]:
import os
import pickle
import numpy as np
import pandas as pd
from scipy import sparse
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from typing import Dict, Tuple

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (
    f1_score, hamming_loss, accuracy_score,
    average_precision_score, precision_recall_curve
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Versions:")
import sklearn, sys
print("  Python:", sys.version.split()[0])
print("  scikit-learn:", sklearn.__version__)


Versions:
  Python: 3.14.4
  scikit-learn: 1.8.0


In [5]:
# ── 1. Data loading ───────────────────────────────────────────────────────────

# ── Config ────────────────────────────────────────────────────────────────────
PKL_PATH   = "/work/gr-fe/bryan/data/SHCS/03_final/mogsage_gnn_embeddings_0.1.pkl"
THRESHOLD  = 0.5
RANDOM_STATE = 42

# ── 1. Load embeddings ────────────────────────────────────────────────────────
with open(PKL_PATH, "rb") as f:
    blob = pickle.load(f)

H         = blob["x"]          # (N, d)
train_idx = blob["train_idx"]
val_idx   = blob["val_idx"]
test_idx  = blob["test_idx"]
y         = blob["y"].astype(int)       # (N, C) multi-hot
n_labels  = y.shape[1]
label_names = blob['label_names']

# ── 2. Splits & standardisation ───────────────────────────────────────────────

print(f"Embeddings : {H.shape}")
print(f"Labels     : {y.shape}  ({n_labels} classes)")
print(f"Train/Val/Test split: {len(train_idx)} / {len(val_idx)} / {len(test_idx)}")

# ── 2. Build train / test sets (train + val → fit, test → evaluate) ───────────
fit_idx = np.concatenate([train_idx, val_idx])
X_full, y_full = H[fit_idx],    y[fit_idx]
X_tr, Y_tr = H[train_idx],    y[train_idx]
X_val, Y_val = H[val_idx],    y[val_idx]
X_te,  Y_te  = H[test_idx],   y[test_idx]

print("Shapes:")
print("  Train:", X_tr.shape, Y_tr.shape)
print("  Val:  ", X_val.shape, Y_val.shape)
print("  Test: ", X_te.shape,  Y_te.shape)

# Sanity checks
def label_stats(Y):
    return {
        "cardinality": float(Y.sum(axis=1).mean()),
        "density": float(Y.sum(axis=1).mean() / Y.shape[1]),
        "positives": Y.sum(axis=0).astype(int)
    }

print("Train stats:", label_stats(Y_tr))
print("Val stats:  ", label_stats(Y_val))
print("Test stats: ", label_stats(Y_te))

Embeddings : (2078, 5792)
Labels     : (2078, 5)  (5 classes)
Train/Val/Test split: 1246 / 415 / 417
Shapes:
  Train: (1246, 5792) (1246, 5)
  Val:   (415, 5792) (415, 5)
  Test:  (417, 5792) (417, 5)
Train stats: {'cardinality': 1.3097913322632424, 'density': 0.26195826645264847, 'positives': array([318, 299, 263, 518, 234])}
Val stats:   {'cardinality': 1.2867469879518072, 'density': 0.25734939759036146, 'positives': array([111,  93,  76, 161,  93])}
Test stats:  {'cardinality': 1.2565947242206235, 'density': 0.2513189448441247, 'positives': array([ 84, 106,  79, 180,  75])}


In [6]:
def evaluate_from_scores(y_true: np.ndarray, y_scores: np.ndarray, thr=0.5) -> Dict[str, float]:
    y_pred = (y_scores >= thr).astype(int)
    return {
        "F1-micro":    f1_score(y_true, y_pred, average="micro", zero_division=0),
        "F1-macro":    f1_score(y_true, y_pred, average="macro", zero_division=0),
        "mAP-micro":   average_precision_score(y_true, y_scores, average="micro"),
        "mAP-macro":   average_precision_score(y_true, y_scores, average="macro"),
        "HammingLoss": hamming_loss(y_true, y_pred),
        "SubsetAcc":   accuracy_score(y_true, y_pred),
        "PredCard":    float(y_pred.sum(axis=1).mean()),
    }

def fit_perlabel_thresholds(y_true: np.ndarray, y_scores: np.ndarray, default=0.5) -> np.ndarray:
    t = np.full(y_true.shape[1], default, dtype=np.float32)
    for j in range(y_true.shape[1]):
        yj = y_true[:, j]
        if yj.sum() == 0:
            continue
        p, r, thr = precision_recall_curve(yj, y_scores[:, j])
        f1 = 2 * p[:-1] * r[:-1] / (p[:-1] + r[:-1] + 1e-12)
        if np.any(np.isfinite(f1)):
            t[j] = thr[np.nanargmax(f1)]
    return t

def find_global_threshold(y_true: np.ndarray, y_scores: np.ndarray,
                          t_min=0.05, t_max=0.8, steps=60, metric="f1_micro") -> Tuple[float, Dict[str,float]]:
    best_t, best_metrics = 0.5, None
    best_val = -np.inf
    grid = np.linspace(t_min, t_max, steps)
    for t in grid:
        m = evaluate_from_scores(y_true, y_scores, thr=t)
        val = m["F1-micro"] if metric == "f1_micro" else m["mAP-micro"]
        if val > best_val:
            best_val, best_t, best_metrics = val, float(t), m
    return best_t, best_metrics

def print_metrics(title: str, m: Dict[str, float]):
    print(title)
    for k in ["F1-micro","F1-macro","mAP-micro","mAP-macro","HammingLoss","SubsetAcc","PredCard"]:
        if k in m:
            print(f"  {k:>11}: {m[k]:.4f}")


In [7]:
# Model: One-vs-Rest Logistic Regression with StandardScaler
def build_model(C=1.0, max_iter=2000, class_weight="balanced"):
    base = LogisticRegression(
        solver="liblinear", penalty="l2", C=C, max_iter=max_iter,
        class_weight=class_weight, random_state=RANDOM_STATE
    )
    return make_pipeline(StandardScaler(with_mean=True, with_std=True),
                         OneVsRestClassifier(base, n_jobs=-1))

# Small grid over C using validation micro-F1 at threshold 0.5
C_grid = [0.1, 0.5, 1.0, 2.0, 5.0]
val_scores = []

for C in C_grid:
    clf = build_model(C=C)
    clf.fit(X_tr, Y_tr)
    # Validation probabilities
    Y_val_scores = clf.predict_proba(X_val)
    m = evaluate_from_scores(Y_val, Y_val_scores, thr=0.5)
    val_scores.append((C, m["F1-micro"], m["mAP-micro"]))
    print_metrics(f"C={C} — Val metrics @thr=0.5", m)

# Pick best C by val micro-F1 (you could switch to mAP if you prefer)
best_C = max(val_scores, key=lambda x: x[1])[0]
print("\nBest C by val micro-F1:", best_C)


C=0.1 — Val metrics @thr=0.5
     F1-micro: 0.4853
     F1-macro: 0.4728
    mAP-micro: 0.4908
    mAP-macro: 0.4851
  HammingLoss: 0.3046
    SubsetAcc: 0.1904
     PredCard: 1.6723
C=0.5 — Val metrics @thr=0.5
     F1-micro: 0.4775
     F1-macro: 0.4646
    mAP-micro: 0.4861
    mAP-macro: 0.4806
  HammingLoss: 0.3027
    SubsetAcc: 0.1976
     PredCard: 1.6096
C=1.0 — Val metrics @thr=0.5
     F1-micro: 0.4740
     F1-macro: 0.4609
    mAP-micro: 0.4848
    mAP-macro: 0.4792
  HammingLoss: 0.3027
    SubsetAcc: 0.2000
     PredCard: 1.5904
C=2.0 — Val metrics @thr=0.5
     F1-micro: 0.4727
     F1-macro: 0.4602
    mAP-micro: 0.4837
    mAP-macro: 0.4778
  HammingLoss: 0.3022
    SubsetAcc: 0.2000
     PredCard: 1.5783
C=5.0 — Val metrics @thr=0.5
     F1-micro: 0.4726
     F1-macro: 0.4600
    mAP-micro: 0.4816
    mAP-macro: 0.4757
  HammingLoss: 0.3012
    SubsetAcc: 0.2024
     PredCard: 1.5687

Best C by val micro-F1: 0.1


In [8]:
# Refit best model on the same train split to produce scores for threshold tuning
best_clf = build_model(C=best_C)
best_clf.fit(X_tr, Y_tr)
Y_val_scores = best_clf.predict_proba(X_val)

# 6a) Global threshold tuned for micro-F1
t_global, m_val_global = find_global_threshold(Y_val, Y_val_scores, metric="f1_micro")
print_metrics(f"\nVal metrics — tuned global threshold t={t_global:.3f}", m_val_global)

# 6b) Per-label thresholds tuned for F1
t_perlabel = fit_perlabel_thresholds(Y_val, Y_val_scores, default=0.5)
Y_val_pred_pl = (Y_val_scores >= t_perlabel).astype(int)
m_val_pl = {
    "F1-micro":  f1_score(Y_val, Y_val_pred_pl, average="micro", zero_division=0),
    "F1-macro":  f1_score(Y_val, Y_val_pred_pl, average="macro", zero_division=0),
    "mAP-micro": average_precision_score(Y_val, Y_val_scores, average="micro"),
    "mAP-macro": average_precision_score(Y_val, Y_val_scores, average="macro"),
    "HammingLoss": hamming_loss(Y_val, Y_val_pred_pl),
    "SubsetAcc":   accuracy_score(Y_val, Y_val_pred_pl),
    "PredCard":    float(Y_val_pred_pl.sum(axis=1).mean()),
}
print_metrics("Val metrics — tuned per-label thresholds", m_val_pl)

# Decide which thresholding strategy to use for final test evaluation
use_perlabel = m_val_pl["F1-micro"] >= m_val_global["F1-micro"]
chosen = "per-label" if use_perlabel else f"global (t={t_global:.3f})"
print("\nChosen thresholding for test:", chosen)



Val metrics — tuned global threshold t=0.279
     F1-micro: 0.5096
     F1-macro: 0.4997
    mAP-micro: 0.4908
    mAP-macro: 0.4851
  HammingLoss: 0.3441
    SubsetAcc: 0.1422
     PredCard: 2.2217
Val metrics — tuned per-label thresholds
     F1-micro: 0.5311
     F1-macro: 0.5161
    mAP-micro: 0.4908
    mAP-macro: 0.4851
  HammingLoss: 0.3311
    SubsetAcc: 0.1373
     PredCard: 2.2434

Chosen thresholding for test: per-label


In [9]:
# Refit best model on the same train split to produce scores for threshold tuning
best_clf = build_model(C=best_C)
best_clf.fit(X_tr, Y_tr)
Y_val_scores = best_clf.predict_proba(X_val)

# 6a) Global threshold tuned for micro-F1
t_global, m_val_global = find_global_threshold(Y_val, Y_val_scores, metric="f1_micro")
print_metrics(f"\nVal metrics — tuned global threshold t={t_global:.3f}", m_val_global)

# 6b) Per-label thresholds tuned for F1
t_perlabel = fit_perlabel_thresholds(Y_val, Y_val_scores, default=0.5)
Y_val_pred_pl = (Y_val_scores >= t_perlabel).astype(int)
m_val_pl = {
    "F1-micro":  f1_score(Y_val, Y_val_pred_pl, average="micro", zero_division=0),
    "F1-macro":  f1_score(Y_val, Y_val_pred_pl, average="macro", zero_division=0),
    "mAP-micro": average_precision_score(Y_val, Y_val_scores, average="micro"),
    "mAP-macro": average_precision_score(Y_val, Y_val_scores, average="macro"),
    "HammingLoss": hamming_loss(Y_val, Y_val_pred_pl),
    "SubsetAcc":   accuracy_score(Y_val, Y_val_pred_pl),
    "PredCard":    float(Y_val_pred_pl.sum(axis=1).mean()),
}
print_metrics("Val metrics — tuned per-label thresholds", m_val_pl)

# Decide which thresholding strategy to use for final test evaluation
use_perlabel = m_val_pl["F1-micro"] >= m_val_global["F1-micro"]
chosen = "per-label" if use_perlabel else f"global (t={t_global:.3f})"
print("\nChosen thresholding for test:", chosen)



Val metrics — tuned global threshold t=0.279
     F1-micro: 0.5096
     F1-macro: 0.4997
    mAP-micro: 0.4908
    mAP-macro: 0.4851
  HammingLoss: 0.3441
    SubsetAcc: 0.1422
     PredCard: 2.2217
Val metrics — tuned per-label thresholds
     F1-micro: 0.5311
     F1-macro: 0.5161
    mAP-micro: 0.4908
    mAP-macro: 0.4851
  HammingLoss: 0.3311
    SubsetAcc: 0.1373
     PredCard: 2.2434

Chosen thresholding for test: per-label


In [11]:
# Final refit on train+val with best C
X_tr_full = X_full
Y_tr_full = y_full

final_clf = build_model(C=best_C)
final_clf.fit(X_tr_full, Y_tr_full)

# Test probabilities
Y_te_scores = final_clf.predict_proba(X_te)

# Use chosen thresholds
if use_perlabel:
    thr_for_test = t_perlabel
    m_test = evaluate_from_scores(Y_te, Y_te_scores, thr=thr_for_test)
else:
    thr_for_test = t_global
    m_test = evaluate_from_scores(Y_te, Y_te_scores, thr=thr_for_test)

print_metrics("\nTest metrics — chosen thresholding", m_test)

# Also show t=0.5 for reference
m_test_05 = evaluate_from_scores(Y_te, Y_te_scores, thr=0.5)
print_metrics("\nTest metrics — threshold=0.5 (reference)", m_test_05)



Test metrics — chosen thresholding
     F1-micro: 0.4928
     F1-macro: 0.4652
    mAP-micro: 0.4453
    mAP-macro: 0.4296
  HammingLoss: 0.3386
    SubsetAcc: 0.1367
     PredCard: 2.0815

Test metrics — threshold=0.5 (reference)
     F1-micro: 0.4694
     F1-macro: 0.4481
    mAP-micro: 0.4453
    mAP-macro: 0.4296
  HammingLoss: 0.2906
    SubsetAcc: 0.2134
     PredCard: 1.4820


In [12]:
def per_class_metrics_from_scores(y_pred: np.ndarray,
                                 y_true: np.ndarray,
                                 threshold: float = 0.5,
                                 assume_one_hot_true: bool = False):
    """
    Compute per-class accuracy, F1 (micro), and F1 (macro) for multi-label or one-vs-rest framing.

    Parameters
    ----------
    y_pred : (H, C) array
        Predicted scores/probabilities (or already-binary predictions).
    y_true : (H, C) array
        Ground-truth labels. If `assume_one_hot_true=True`, it should be one-hot (single-label).
        Otherwise it can be multi-hot (multi-label).
    threshold : float
        Threshold used to binarize y_pred if it is not already {0,1}.
    assume_one_hot_true : bool
        If True, y_true is treated as one-hot and will be converted to binary per class naturally.
        If False, y_true is assumed already binary multi-label (0/1).

    Returns
    -------
    metrics : dict
        {
          "per_class": {
             "accuracy": (C,),
             "f1_micro": (C,),
             "f1_macro": (C,),
             "precision": (C,),
             "recall": (C,),
             "support_pos": (C,),  # number of positives in y_true per class
          },
          "confusion": {
             "tp": (C,), "fp": (C,), "fn": (C,), "tn": (C,)
          }
        }

    Notes
    -----
    For each class k, we treat it as a binary problem (class k vs not-k):
      TP_k, FP_k, FN_k, TN_k
    Per-class micro-F1 equals the binary F1 for that class:
      F1_micro_k = 2*TP_k / (2*TP_k + FP_k + FN_k)
    Per-class macro-F1 is the average of positive-class F1 and negative-class F1:
      F1_macro_k = (F1_pos_k + F1_neg_k)/2
    """
    y_pred = np.asarray(y_pred)
    y_true = np.asarray(y_true)

    if y_pred.shape != y_true.shape or y_pred.ndim != 2:
        raise ValueError(f"Expected y_pred and y_true to have same shape (H, C). "
                         f"Got {y_pred.shape=} and {y_true.shape=}.")

    H, C = y_true.shape

    # Binarize predictions if needed
    if np.issubdtype(y_pred.dtype, np.floating) or np.any((y_pred != 0) & (y_pred != 1)):
        y_pred_bin = (y_pred >= threshold).astype(np.int64)
    else:
        y_pred_bin = y_pred.astype(np.int64)

    # Ensure y_true is binary
    if assume_one_hot_true:
        # Allow one-hot ints or floats; treat >0 as 1
        y_true_bin = (y_true > 0).astype(np.int64)
    else:
        y_true_bin = y_true.astype(np.int64)
        if np.any((y_true_bin != 0) & (y_true_bin != 1)):
            raise ValueError("y_true must be binary (0/1) when assume_one_hot_true=False.")

    # Confusion terms per class (vectorised)
    tp = np.sum((y_pred_bin == 1) & (y_true_bin == 1), axis=0)
    fp = np.sum((y_pred_bin == 1) & (y_true_bin == 0), axis=0)
    fn = np.sum((y_pred_bin == 0) & (y_true_bin == 1), axis=0)
    tn = np.sum((y_pred_bin == 0) & (y_true_bin == 0), axis=0)

    # Per-class accuracy
    acc = (tp + tn) / (tp + fp + fn + tn)

    # Precision/Recall for positive class (per class)
    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp, dtype=float), where=(tp + fp) != 0)
    recall    = np.divide(tp, tp + fn, out=np.zeros_like(tp, dtype=float), where=(tp + fn) != 0)

    # Per-class F1 (micro == binary F1 for this one-vs-rest class)
    f1_micro = np.divide(
        2 * tp, 2 * tp + fp + fn,
        out=np.zeros_like(tp, dtype=float),
        where=(2 * tp + fp + fn) != 0
    )

    # Negative-class F1 (treat "not class k" as positive), then macro-average within the class.
    precision_neg = np.divide(tn, tn + fn, out=np.zeros_like(tn, dtype=float), where=(tn + fn) != 0)
    recall_neg    = np.divide(tn, tn + fp, out=np.zeros_like(tn, dtype=float), where=(tn + fp) != 0)
    f1_neg = np.divide(
        2 * precision_neg * recall_neg, precision_neg + recall_neg,
        out=np.zeros_like(precision_neg, dtype=float),
        where=(precision_neg + recall_neg) != 0
    )

    f1_macro = 0.5 * (f1_micro + f1_neg)

    return {
        "per_class": {
            "accuracy": acc,
            "f1_micro": f1_micro,
            "f1_macro": f1_macro,
            "precision": precision,
            "recall": recall,
            "support_pos": np.sum(y_true_bin == 1, axis=0),
        },
        "confusion": {"tp": tp, "fp": fp, "fn": fn, "tn": tn},
    }


# Example usage:
# metrics = per_class_metrics_from_scores(y_pred, y_true, threshold=0.5)
# print(metrics["per_class"]["accuracy"])  # shape (C,)
# print(metrics["per_class"]["f1_micro"])  # shape (C,)
# print(metrics["per_class"]["f1_macro"])  # shape (C,)


In [14]:
per_class_metrics_from_scores((Y_te_scores > 0.5)*1 , Y_te)

{'per_class': {'accuracy': array([0.77218225, 0.71942446, 0.71702638, 0.60671463, 0.73141487]),
  'f1_micro': array([0.5026178 , 0.45070423, 0.37234043, 0.56613757, 0.34883721]),
  'f1_macro': array([0.67743643, 0.63114921, 0.59483894, 0.60324422, 0.58982646]),
  'precision': array([0.44859813, 0.44859813, 0.32110092, 0.54040404, 0.30927835]),
  'recall': array([0.57142857, 0.45283019, 0.44303797, 0.59444444, 0.4       ]),
  'support_pos': array([ 84, 106,  79, 180,  75])},
 'confusion': {'tp': array([ 48,  48,  35, 107,  30]),
  'fp': array([59, 59, 74, 91, 67]),
  'fn': array([36, 58, 44, 73, 45]),
  'tn': array([274, 252, 264, 146, 275])}}

In [16]:
(Y_te_scores > 0.5)*1 

array([[0, 0, 1, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0],
       ...,
       [0, 1, 1, 0, 1],
       [0, 0, 0, 1, 0],
       [0, 0, 1, 0, 0]], shape=(417, 5))

In [ ]:
# Precision–Recall curves (micro + a few frequent labels)
from sklearn.metrics import precision_recall_curve

def plot_pr_curves(y_true, y_scores, label_names, k=6, mode="freq", title_prefix="Test"):
    ap_micro = average_precision_score(y_true, y_scores, average="micro")
    freq = y_true.sum(axis=0).astype(int)
    ap = np.array([average_precision_score(y_true[:, j], y_scores[:, j]) for j in range(y_true.shape[1])])

    if mode == "ap":
        idx = np.argsort(-ap)[:k]
    else:
        idx = np.argsort(-freq)[:k]
    names = [str(label_names[i]) for i in idx]

    plt.figure(figsize=(7, 6))
    p_micro, r_micro, _ = precision_recall_curve(y_true.ravel(), y_scores.ravel())
    plt.plot(r_micro, p_micro, color="black", lw=2, label=f"micro-AP = {ap_micro:.3f}")

    for i, name in zip(idx, names):
        p, r, _ = precision_recall_curve(y_true[:, i], y_scores[:, i])
        plt.plot(r, p, lw=1.2, label=f"{name}")

    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title(f"{title_prefix} PR — micro and {len(idx)} labels")
    plt.legend(loc="lower left", fontsize=8, ncol=2)
    plt.grid(True, alpha=0.3)
    plt.show()

plot_pr_curves(Y_val, Y_val_scores, label_names, k=6, mode="freq", title_prefix="Val")
plot_pr_curves(Y_te,  Y_te_scores,  label_names, k=6, mode="freq", title_prefix="Test")

# Per-label AP and F1 bar chart (top by frequency)
Y_te_pred = (Y_te_scores >= (t_perlabel if use_perlabel else t_global)).astype(int)
per_f1 = f1_score(Y_te, Y_te_pred, average=None, zero_division=0)
per_ap = np.array([average_precision_score(Y_te[:, j], Y_te_scores[:, j]) for j in range(Y_te.shape[1])])
freq = Y_te.sum(axis=0).astype(int)

idx = np.argsort(-freq)[:10]
names = [str(label_names[i]) for i in idx]
x = np.arange(len(idx)); w = 0.4
plt.figure(figsize=(10, 4))
plt.bar(x - w/2, per_ap[idx], width=w, label="AP")
plt.bar(x + w/2, per_f1[idx], width=w, label="F1")
plt.xticks(x, names, rotation=45, ha="right")
plt.ylabel("Score")
plt.title("Top-10 labels by test frequency — AP and F1")
plt.legend(); plt.tight_layout()
plt.show()